# Iterator

In [1]:
import random
from collections.abc import Iterable, Iterator, Sequence
from contextlib import suppress
from typing import Any, NoReturn, Self, TypeVar

## Intro

Iterator is a behavioural design pattern. It allows to traverse a collection without revealing the internal structure of the latter. In lieu of embedding possible ways to iterate over a collection, you may define iterators each responsible for a certain traversion **behaviour**.

In Python there are [iterable](https://docs.python.org/3/glossary.html#term-iterable) objects that are capable of returning its members one at a time. Examples are all sequence types like [list](https://docs.python.org/3/library/stdtypes.html#list), [str](https://docs.python.org/3/library/stdtypes.html#str) and some non-sequence types like [dict](https://docs.python.org/3/library/stdtypes.html#dict), [file(-like) objects](https://docs.python.org/3/glossary.html#term-file-object) and so forth.

A class is iterable if:

- inherits from the [Iterable](https://docs.python.org/3/library/collections.abc.html#collections.abc.Iterable) class and has to define the abstract [\_\_iter\_\_()](https://docs.python.org/3/library/stdtypes.html#container.__iter__) method - this is an example of [nominal subtyping](https://typing.python.org/en/latest/reference/protocols.html) (via inheritance);
- just defines the [\_\_iter\_\_()](https://docs.python.org/3/library/stdtypes.html#container.__iter__) method - this is an example of [strucutural subtyping](https://typing.python.org/en/latest/reference/protocols.html) (when defining methods (and maybe attibutes) that match the structure of Iterable protocol)the Iterable protocol)
- supports the [Sequence](https://docs.python.org/3/library/collections.abc.html#collections.abc.Sequence) protocol via either nominal or structural subtyping (to be covered later).

Let's just see all of this in use.

In [2]:
class SubIterable(Iterable):  # nominal subtyping - you see the "Iterable" name
    def __iter__(self) -> Self:
        return self

class SupportsIter:  # structural subtyping - you see the __iter__() dunder defined
    def __iter__(self) -> Self:  # DUNDER = Double Underscore
        return self

subit = SubIterable()
supit = SupportsIter()

# The objects are `Iterable`s -> they do have the `__iter__()` dunder defined
assert isinstance(subit, Iterable)
assert isinstance(supit, Iterable)

# The objects are NOT `Iterator`s -> they do NOT have the `__next__()` dunder defined
assert not isinstance(subit, Iterator)
assert not isinstance(supit, Iterator)

# Because the defined `__iter__()` methods does not return iterators,
# calling `iter(obj)` failes.

with suppress(TypeError):
    iter(subit)  # like `subit.__iter__()`

with suppress(TypeError):
    supit.__iter__()  # like `iter(supit)`

# Because `iter(obj)` does not return an iterator,
# these objects cannot be used in a `for` loop
# because it relies on the `iterable.__next__()`/`next(iterable)` interface

with suppress(TypeError):
    for _ in subit:
        break

with suppress(TypeError):
    for _ in supit:
        break

An interesting situation we have: objects ARE iterable, but they cannot be used properly in a [for](https://docs.python.org/3/reference/compound_stmts.html#for) loop. A key to understand this problem is to remember that a `for` loop, for example, `lst = [elem for elem in iterable]` is roughly equivalent to the following set of steps:

1. `_iterator = iter(iterable)` - call `iter` on an `iterable` object (that is why it needs to implement the **\_\_iter\_\_()** dunder method) and store the returned iterator object in an internal variable. e.g., `_iterator`
2. `while True:` - going in the loop
3. `elem = next(_iterator)` - the `_iterator` nust implement the **\_\_next\_\_()** dunder method and give back an item from the `iterable` object
4. keep going until the `StopIteration` is raised -> the iterations are over

In [3]:
tup = (1, 2, 3, 4)  # a tuple is a Sequence
assert isinstance(tup, Sequence)
# Sequence protocol -> __getitem__(self, key) (e.g., `tup[0]`) and __len__(self) (e.g., `len(tup)`)
assert hasattr(tup, '__getitem__')
assert hasattr(tup, '__len__')

print(f"tuple if forable = {list(el for el in tup)}")

# The above is something like

tup_iterator = iter(tup)
print(
    f"Iterator = {tup_iterator}:",
    "\n".join(
        [
            f"type={type(tup_iterator)}",
            f"is_iterable={isinstance(tup_iterator, Iterable)}",
            f"is_iterator={isinstance(tup_iterator, Iterator)}",
            f"has __iter__? {hasattr(tup_iterator, '__iter__')}",
            f"has __next__? {hasattr(tup_iterator, '__next__')}",
        ]
    ),
    sep="\n",
    end="\n\n",
)

while True:
    try:
        item = next(tup_iterator)  # like tup_iterator.__next__()
    except StopIteration as e:
        print(f"Iteration is over: {e}")
        break

    print(f"Item = {item}")

tuple if forable = [1, 2, 3, 4]
Iterator = <tuple_iterator object at 0x750b5bd38e50>:
type=<class 'tuple_iterator'>
is_iterable=True
is_iterator=True
has __iter__? True
has __next__? True

Item = 1
Item = 2
Item = 3
Item = 4
Iteration is over: 


So, a tuple is:

- a Sequence -> defines \_\_getitem\_\_(key) and \_\_len\_\_()
- an Iterable -> defines \_\_iter\_\_()
- an Iterator -> that is not so simple to conclude for now

Let's define a class that can be used in a `for` loop safely.

In [4]:
class MyCounter(Iterable):
    """This class is Iterable and Iterator per se."""

    def __init__(self, start: int = 5) -> None:
        self._countdown = start

    def __iter__(self) -> Self:
        # our iterator is reusable
        self._counter = self._countdown
        return self

    def __next__(self) -> int:
        self._counter -= 1
        if self._counter < 0:
            # the message is not necessary
            msg = f"EOC: {self._counter}"
            # raising just StopIteration is enough
            raise StopIteration(msg)
        return self._counter


counter = MyCounter(start=5)

assert isinstance(counter, Iterable)
assert isinstance(counter, Iterator)

iter(counter)  # OK

# does these `iter` and `next` calls for us
for c in counter:
    print(f"count = {c}")
print()

# # checking that the MyCounter is resuable 
# for c in counter:
#     print(f"count = {c}")
# print()

iterator = iter(counter)
while True:
    try:
        c = next(counter)
    except StopIteration as e:
        print(f"Stop: message={e}")
        break
    print(f"count = {c}")

count = 4
count = 3
count = 2
count = 1
count = 0

count = 4
count = 3
count = 2
count = 1
count = 0
Stop: message=EOC: -1


**Please be careful now - important notes go here.**

What if we separate the concerns:

1. define a class, so it be "MyIterable", with only the **\_\_iter\_\_()** dunder (double underscore) method implemented;
2. define a class, so it be "MyIterator", with only the **\_\_next\_\_()** dunder method implemented;

What is desirable to attain? The `iter(MyIterable)` must return an instance of the "MyIterator" class that can give elements from the "MyIterable" instance.

What is expected? An instance of "MyIterable" is `for`able.

In [5]:
class MyIterator:
    def __init__(self, seq: Sequence) -> None:
        self._seq = seq
        self._idx = 0

    def __next__(self) -> Any:
        try:
            idx = self._idx
            self._idx += 1  # for the sake of proceeding
            return self._seq[idx]
        except IndexError:
            raise StopIteration

class MyIterable:
    def __init__(self, it: Iterable | None = None) -> None:
        self._lst = list(it or [])

    def __iter__(self) -> MyIterator:
        return MyIterator(seq=self._lst)


# IT WORKS!
for item in MyIterable(range(10)):
    print(f"item = {item}")


my_it = MyIterable()
my_iter = iter(my_it)

# MyIterable is Iterable -> OK and the __iter__ is defined
assert isinstance(my_it, Iterable)
# MyIterable is NOT Iterator -> OK and no __next__ is defined
assert not isinstance(my_iter, Iterator)

# MyIterator is NOT Iterable -> NO __iter__ defined
assert not isinstance(my_iter, Iterable)
# WHAT?!
# MyIterator is NOT Iterator but the __next__ defined
assert not isinstance(my_iter, Iterator)

iter(my_it)  # OK
with suppress(TypeError):
    iter(my_iter)  # no __iter__

item = 0
item = 1
item = 2
item = 3
item = 4
item = 5
item = 6
item = 7
item = 8
item = 9


It works and the results are fascinating!

The "MyIterable" is suitable for a `for` loop -> no TypeError, no other complaints, iterations go fine. But we have to examine the assertions which tell us that:

1. "MyIterable" IS "Iterable" by implementing the \_\_iter\_\_() method.
2. "MyIterable" IS NOT "Iterator" because it DOES NOT implement the \_\_next\_\_() method.
3. "MyIterator" IS NOT "Iterable" because it DOES NOT implement the \_\_iter\_\_() - this is fine (for now) because **an iterator does not need to be iterable itself** - the purpose of an iterator is to know how to traverse over an associated iterable object.
4. "MyIterator" IS NOT "Iterator" and this IS CREEPY YET UNDERSTANDABLE and gives us a clue that IN PYTHON, and `Iterator` IS a sub(type/class) of `Iterable`

Let's "reiterate".

IN GENERAL:

- An iterable object is an object that can be travesed/iterated, i.e., its elements can be retrieved.
- An iterator can retrieve items/elements/etc from an iterable object. An iterator IS NOT BOUND to be an iterable object itself.

IN PYTHON:

- the `Iterator` type IS `Iterable`, but not vice versa -> an instance of the `Iterator` class must be iterable per se and implement the `__iter__()/__next__()` pair either nominally (inheritance from the `Iterator`) or structurally.

Why so? I guess it can be explained like so:
> If an iterator "knows" how to get items from an iterable object, it is like iterating over this iterator itself!

I gather it is also for the reason of flexibility so the possible code would not panic:
```python
[_ for _ in iterable]  # OK
_iterator = iter(iterable)
[_ for _ in _iterator]  # does not crash and it is like [_ for _ in iter(iterable)] (which is redundant)
```

In [6]:
class DummyIterator:
    def __iter__(self) -> Self:
        return self

    def __next__(self) -> NoReturn:
        raise StopIteration

diter = DummyIterator()

assert isinstance(diter, Iterable)
assert isinstance(diter, Iterator)

assert issubclass(Iterator, Iterable)

[_ for _ in diter]
_iterator = iter(diter)
[_ for _ in _iterator];  # this `;` is to silence the cell output - Jupyter guts

Iterable-Iterator relationship (Plant)UML diagramm:

![Iterable-Iterator](./Iterable-Iterator.png)

## Sequences

The objects you can iterate over may be defined not only through the \_\_iter\_\_()/\_\_next\_\_() pair. If a class defines the [\_\_getitem\_\_(key: int | slice)](https://docs.python.org/3/reference/datamodel.html#object.__getitem__) and [\_\_len\_\_()](https://docs.python.org/3/reference/datamodel.html#object.__len__) dunder (double underscore) methods, then it is considered as a [sequence](https://docs.python.org/3/glossary.html#term-sequence) and thus becomes an iterable object.

In [7]:
class DummySequence:
    def __init__(self, seq: Iterable) -> None:
        self._seq = tuple(seq)

    def __getitem__(self, key: int | slice) -> Any:
        res = self._seq[key]
        if isinstance(key, int):
            return res
        return type(self)(res)

    def __len__(self) -> int:
        return len(self._seq)

    def __repr__(self) -> str:
        return f"{type(self).__name__}(seq={self._seq})"

seq = DummySequence([1, 4, 6, 9])

print(f"seq[1::2] = {seq[1::2]}")
print(f"len(seq) = {len(seq)}")

assert not isinstance(seq, Iterable)  # no __iter__

# https://docs.python.org/3/glossary.html#term-sequence
# IMPORTANT and UNEXPECTED!!!!
assert not isinstance(seq, Sequence)

for elem in seq:
    print(f"Element = {elem}")

seq[1::2] = DummySequence(seq=(4, 9))
len(seq) = 4
Element = 1
Element = 4
Element = 6
Element = 9


How about the `isinstance(obj, Sequence)` check? It fails regardless the Sequence protocol methods have been defined. You can investigate and start from the following answers:

- https://stackoverflow.com/questions/64654517/why-does-isinstance-check-for-abc-sequence-return-false-for-custom-classes
- https://stackoverflow.com/questions/76998232/the-implementation-of-sequence-interface-is-not-sufficient-to-be-a-sequence

A way to discouver it by example is to define a custom class that inherits from the `collections.abc.Sequence` class. The latter has abstract methods which must be overriden in a child class, Python will alert about them.

In [8]:
class MySequence(Sequence):
    def __init__(self, it: Iterable | None = None) -> None:
        self._seq = tuple(it or [])

    # sequence protocol
    def __getitem__(self, key: int | slice) -> Any:
        res = self._seq[key]
        if isinstance(key, int):
            return res
        return type(self)(res)

    # sequence protocol
    def __len__(self) -> int:
        return len(self.seq)

seq = MySequence()

# inheritance SAVES the situation
assert isinstance(seq, Sequence)

for item in MySequence([1, 2, 3]):
    print(f"Elem = {item}")

Elem = 1
Elem = 2
Elem = 3


Consider reading more about the [abc.Sequence](https://docs.python.org/3/library/collections.abc.html#collections.abc.Sequence) class.

## Iterator Design Pattern

Suppose we have some data structure that can be iterable not only in just one way. Imagine we have a 2D array (or list - whatever). A usual way is to walk through it row by row.

In [9]:
matrix = [[1], [2, 3], [4, 5, 6]]

# no need for an iterator here
for row in matrix:
    print(f"Row = {row}")

Row = [1]
Row = [2, 3]
Row = [4, 5, 6]


What if we need to iterate over its columns? Let's write a function for this.

In [10]:
T = TypeVar("T")

def iter_over_columns(mtx: list[list[T]]) -> Iterator[list[T]]:
    for ridx in range(len(mtx)):
        column: list[T] = []
        for row in mtx:
            if not row:
                continue

            item = None
            try:
                item = row[ridx]
            except IndexError:
                pass
            column.append(item)

        yield column
        

matrix = [[1], [2, 3], [4, 5, 6]]

# The `iter_over_columns` function is an iterator by nature,
# but it will fail the `isinstance` test on being Iterator.
for col in iter_over_columns(matrix):
    print(f"Column = {col}")

assert not isinstance(iter_over_columns, Iterator)

Column = [1, 2, 4]
Column = [None, 3, 5]
Column = [None, None, 6]


And what if we need other ways of traversion? It is doable with more iterators.

In [11]:
U = TypeVar("U")


class Matrix:
    def __init__(self, it: Iterable[Iterable[U]] | None = []) -> None:
        self._mtx: list[list[U]] = [list(row) for row in it] if it else []

    def __iter__(self) -> Iterator[U]:
        """The default strategy per rows."""

        for row in self._mtx:
            yield row

    def __reversed__(self) -> Iterator[U]:
        """The default strategy per rows."""

        for row in reversed(self._mtx):
            yield row


class MatrixColumnWalker:
    def __init__(self, matrix: Matrix) -> None:
        self._mtx = matrix

    def __iter__(self) -> Self:
        self.__it = self._traverse()
        return self

    def _traverse(self) -> Iterator[list[U]]:
        for ridx in range(len(matrix)):
            column: list[U] = []
            for row in self._mtx:
                if not row:
                    continue

                item = None
                try:
                    item = row[ridx]
                except IndexError:
                    pass
                column.append(item)

            yield column

    def __next__(self) -> Iterator[list[U]]:
        return next(self.__it)


class MatrixClockwiseWalker:
    """Up to you."""


class MatrixClounterClockwiseWalker:
    """Up to you."""


for matrix in (
    [],
    [[], []],
    [[1, 2], [3, 4]],
    [[1], [2, 3], [4, 5, 6]]  # not a good matrix
):
    print(f"Matrix: {matrix}")
    print(f"Per rows: {[row for row in matrix]}")
    print(f"Per rows reversed: {[row for row in reversed(matrix)]}")

    print(f"Per columns: {[row for row in MatrixColumnWalker(matrix)]}")

    print()

Matrix: []
Per rows: []
Per rows reversed: []
Per columns: []

Matrix: [[], []]
Per rows: [[], []]
Per rows reversed: [[], []]
Per columns: [[], []]

Matrix: [[1, 2], [3, 4]]
Per rows: [[1, 2], [3, 4]]
Per rows reversed: [[3, 4], [1, 2]]
Per columns: [[1, 3], [2, 4]]

Matrix: [[1], [2, 3], [4, 5, 6]]
Per rows: [[1], [2, 3], [4, 5, 6]]
Per rows reversed: [[4, 5, 6], [2, 3], [1]]
Per columns: [[1, 2, 4], [None, 3, 5], [None, None, 6]]



I was lazy enough to implement "Matrix[Counter]ClockWiseIterator"s to demonstrate other ways of traversing the matrix, but it is possible (I guess so). The same laziness did not encourage me to write "Tree" data structures from scratch to demonstrate that, e.g., "DepthFirstSearch" or "BreadthFirstSearch" iterators could be useful when traversing a tree in several ways. Perhaps one day it will be done in my [ADS](https://github.com/stankudrow/ADS/tree/main/python) repository.

## Asynchronous iterators

If there are synchronous ("default") iterators, what about asynchronous ones? They do exist and Python provides you with the interfaces.

Before you go further, could you guess the signatures of methods that can turn a class into an asynchronous iterator? There are some hints:

- these methods are dunders (it's pretty obvious);
- the names of these dunders start with an 'a' prefix letter;
- these methods should (but not necessarily) be alike their synchronouse counterparts.

In [12]:
import asyncio
from collections.abc import AsyncIterable, AsyncIterator

Time to reveal some thruth.

First, an `async def` function with `yield` inside is an asynchronous iterator (therefore, an asynchronous iterable).

In [13]:
async def sourcerer(uplimit: int = 5) -> AsyncIterator[int]:
    """Yields integer values.

    I am a source of integer values, that is why I am a sourcerer.
    """
    for item in range(uplimit):
        await asyncio.sleep(0)  # simulate some I/O-bound work
        yield item


asrc = sourcerer(3)
assert isinstance(asrc, AsyncIterable)
assert isinstance(asrc, AsyncIterator)
assert not isinstance(asrc, Iterable)  # different kind

with suppress(TypeError):  # async_generator object is not iterable
    for i in asrc:
        print("NOT FOR")

# Jupyter "knows" how to run async code
async for ai in asrc:
    print(f"AITEM = {ai}")

AITEM = 0
AITEM = 1
AITEM = 2


Now with classes.

In [14]:
class MyAsyncNexter:
    # expected to be an async
    async def __anext__(self) -> NoReturn:
        raise StopAsyncIteration

class MyAsyncIterable:
    def __init__(self, n: int | None = None) -> None:
        self._source = sourcerer() if n is None else sourcerer(n)

    # Why def???
    def __aiter__(self) -> MyAsyncNexter:
        return MyAsyncNexter()


class MyAsyncIterator:
    def __init__(self, async_source) -> None:
        self._asrc = async_source

    def __aiter__(self) -> Self:
        return self  # because I am an iterator at the same time

    async def __anext__(self) -> NoReturn:
        try:
            return await anext(self._asrc)
        except StopAsyncIteration:
            print("No resources left")
            raise  # the caught exception is reraised


ait = MyAsyncIterable()
assert isinstance(ait, AsyncIterable)
assert not isinstance(ait, AsyncIterator)
assert not isinstance(ait, Iterable)  # different kind

anexter = MyAsyncNexter()
assert not isinstance(anexter, AsyncIterable)  # as expected
assert not isinstance(anexter, AsyncIterator)  # as expected

aiterator = MyAsyncIterator(None)  # a hack only !!!
assert isinstance(aiterator, AsyncIterable)  # as expected
assert isinstance(aiterator, AsyncIterator)  # as expected

async for ai in MyAsyncIterator(sourcerer(10)):  # it is AsyncIterable anyway
    print(f"AITEM = {ai}")

AITEM = 0
AITEM = 1
AITEM = 2
AITEM = 3
AITEM = 4
AITEM = 5
AITEM = 6
AITEM = 7
AITEM = 8
AITEM = 9
No resources left


You see, the `__aiter__()` dunder is `def`, NOT `async def`!

I think this makes sense and there are some reasons for that:

- like `iter(it)`, calling `aiter(ait)` must return an asynchronous iterator
- it is pointless to `await` an asynchronous iterator for it can be just returned as it is
- an asynchronous iterator IS RESPONSIBLE for conveying items/resources/... which can be awaited 

## Summary

Iterators are cool when:

- There are more than one way to traverse some collection. A default strategy may be implemented in the `__iter__()` method and a default backward strategy, if needed, in the `__reversed__()` one.
- Iterators may be just functions, no need to make them classes.
- You don't need to overload a collection, you can just define an iterator that will make the collection to behave in the way you want to get the items. 

For curious.

You can make your inquires about iterators and generators: similarities nad differences. The same sh...things about asynchronous variants.